In [ ]:
# %load_ext autoreload
# %autoreload 2

import sys
sys.path.append('/home/galk/LanguageDynamics/src') 
from data_generation import *
from config import TinyLMConfig, TinyDVAEConfig, TrainingConfig,TrainingDVAEConfig, ExperimentConfig
from models import TinyLlamaTransformer
from models import TransformerAutoencoder, TransformerDVAE, kl_divergence_gaussians
import random
import numpy as np
from tqdm import tqdm

import os
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
# from datasets import load_dataset
from datetime import datetime

from importlib import reload

models_path = '/home/galk/LanguageDynamics/models/linguistic_flip_flop'

In [ ]:
reload(sys.modules['models.autoencoders'])

In [ ]:
### Hyperparameters
E = 1
D = 1
N = 4
M = 2
G = 8
L_E = [1 for _ in range(E)]
L_D = [1 for _ in range(D)]
L_N = [3, 4, 5, 6]
L_M = [1 for _ in range(M)]

special_tokens = []

G_MIN = len(special_tokens) + E + D + M  # special tokens, plus other tokens
G_MAX = G_MIN + G - 1

memory_limit = float("inf")

### Sample noise trajectories
noise_list = []
for n_i in range(N):
    noise_list.append(random.sample(range(G_MIN, G_MAX+1), L_N[n_i]))

vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"G{g+1}" for g in range(G)]
token2id = {tok: idx for idx, tok in enumerate(vocab)}
id2token = {idx: tok for tok, idx in token2id.items()}
vocab_size = len(vocab)

# Non-Terminal (NT) vocab
NT_vocab = special_tokens + [f"E{e+1}" for e in range(E)] + [f"D{d+1}" for d in range(D)] + [f"M{m+1}" for m in range(M)] + [f"N{n+1}" for n in range(N)]
NT_token2id = {tok: idx for idx, tok in enumerate(NT_vocab)}
NT_id2token = {idx: tok for tok, idx in NT_token2id.items()}
NT_vocab_size = len(NT_vocab)

# Initialize NT to T transitions
NT_to_T = {token: token if token not in [f"N{n+1}" for n in range(N)] else [id2token[id] for id in noise_list[[f"N{n+1}" for n in range(N)].index(token)]] for token in NT_vocab }

### Initialize Transition matrices
# P_transitions = np.zeros((E+1, NT_vocab_size, NT_vocab_size))  # [states, source, target]
P_transitions = np.zeros((1, NT_vocab_size, NT_vocab_size))  # [states, source, target]

# zero mode - no memory
# P_transitions[0, NT_token2id['<EOS>'], NT_token2id['<BOS>']] = 1
# P_transitions[0, NT_token2id.<BOS>'], NT_token2id.E1']] = 1/(2*N)
# P_transitions[0, NT_token2id['<BOS>'], NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/N
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(2*N)
P_transitions[0, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[0, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['<EOS>']] = 0.2
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['E1']:NT_token2id[f'E{E}']+1] = 1/(N)
P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 1/(N)
# P_transitions[0, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['<EOS>']] = 1/(2*N)
np.fill_diagonal(P_transitions[0], 0)

# memory mode
# P_transitions[1, NT_token2id['D1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M  ## THIS WILL BE DECIDED BY THE CONTEXT (which M was memorized)
# P_transitions[1, NT_token2id['E1']:NT_token2id[f'E{E}']+1, NT_token2id['M1']:NT_token2id[f'M{M}']+1] = 1/M
# P_transitions[1, NT_token2id['M1']:NT_token2id[f'M{M}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['N1']:NT_token2id[f'N{N}']+1] = 1/(N)
# P_transitions[1, NT_token2id['N1']:NT_token2id[f'N{N}']+1, NT_token2id['D1']:NT_token2id[f'D{D}']+1] = 2/(N)
# np.fill_diagonal(P_transitions[1], 0)


# Normalize Matrices
P_transitions = P_transitions / P_transitions.sum(axis=-1, keepdims=True) # normalize rows to sum to 1
P_transitions = np.nan_to_num(P_transitions) # replace NaNs with 0


In [ ]:
model_config = TinyLMConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03
)

training_config = TrainingConfig(
    lr=1e-4,
    batch_size=256,
    grad_clipping=True
)

experiment_config = ExperimentConfig(
    epochs=500,
    # checkpoint_path=None,
    checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt",
    save_every=1,
    device_index=0,
    model_config=model_config,
    training_config=training_config
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

print(f"Using device: {experiment_config.device}")

In [ ]:
context_window = experiment_config.model_config.context_window
n_train = 1
n_train_windows = 100000
max_steps = n_train_windows * (context_window + 1)
val_ratio = 0.01
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_train, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=int(max_steps*val_ratio), memory_limit=memory_limit)
train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

In [ ]:
print(train_blocks.shape)
print(val_blocks.shape)

In [ ]:
# Create datasets
train_dataset = TokensDataset(train_blocks)
val_dataset = TokensDataset(val_blocks)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=True,
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=experiment_config.training_config.batch_size,
    shuffle=False,  # No need to shuffle validation data
    drop_last=True
)

# Verify the split
print("\nDataLoader Info:")
print(f"Training batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

In [ ]:
# --------- 7. Training --------- WITH CHECKPOINT LOADING 26/06/25
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.epochs = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_losses = metrics_dict['train_losses']
        self.train_accuracies = metrics_dict['train_accuracies']
        self.val_losses = metrics_dict['val_losses']
        self.val_accuracies = metrics_dict['val_accuracies']
        self.epochs = list(range(1, len(self.train_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

        # Plot losses
        ax1.plot(self.epochs, self.train_losses, label='Train Loss')
        ax1.plot(self.epochs, self.val_losses, label='Val Loss')
        ax1.set_xlabel('Epoch')
        ax1.set_ylabel('Loss')
        ax1.set_title('Training and Validation Loss')
        ax1.legend()
        ax1.grid(True)

        # Plot accuracies
        ax2.plot(self.epochs, self.train_accuracies, label='Train Accuracy')
        ax2.plot(self.epochs, self.val_accuracies, label='Val Accuracy')
        ax2.set_xlabel('Epoch')
        ax2.set_ylabel('Accuracy')
        ax2.set_title('Training and Validation Accuracy')
        ax2.legend()
        ax2.grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png")

        plt.show()

def load_checkpoint(checkpoint_path, model, device, optimizer=None):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch and loaded metrics
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    for key in ['vocab_size', 'embed_dim', 'n_layers', 'n_heads', 'ffn_dim', 'context_window']:
        if CONFIG[key] != saved_config[key]:
            raise ValueError(f"Checkpoint config mismatch for {key}: "
                           f"current={CONFIG[key]}, saved={saved_config[key]}")

    starting_epoch = checkpoint['epoch']
    return starting_epoch, checkpoint['metrics']

n_trials = 1
device = experiment_config.device
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'LM':
        model = TinyLlamaTransformer(
            experiment_config.model_config
        ).to(device)

    elif experiment_config.model_config.mode == 'AE':
        model = TransformerAutoencoder(
            experiment_config.model_config
        ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=experiment_config.training_config.lr)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                device,
                optimizer
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0

    print("Starting training loop...")
    global_start = time.time()

    def validate(model, val_loader, criterion, model_config, device):
        model.eval()
        total_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                if model_config.mode == 'LM':
                    logits = model(x)
                elif model_config == 'AE':
                    logits, _, _, _ = model(x, decoding_seed=x[:, :-1])
                loss = criterion(logits.view(-1, len(model_config.vocab)), y.view(-1))
                acc = calculate_accuracy(logits, y, pad_id)
                total_loss += loss.item()
                total_acc += acc
        return total_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        epoch_loss = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, (x, y) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            y = y.to(device)

            if experiment_config.model_config.mode == 'LM':
                logits = model(x)
            elif experiment_config.model_config.mode == 'AE':
                logits, _, _, _ = model(x, decoding_seed=x[:,:-1])
            loss = criterion(logits.view(-1, len(experiment_config.model_config.vocab)), y.view(-1))
            acc = calculate_accuracy(logits, y, pad_id)

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()

            epoch_loss += loss.item()
            epoch_acc += acc

        # Calculate training metrics
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)

        # Validation phase
        val_loss, val_acc = validate(model, val_loader, criterion, experiment_config.model_config, device)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_losses.append(train_loss)
        metrics.train_accuracies.append(train_acc)
        metrics.val_losses.append(val_loss)
        metrics.val_accuracies.append(val_acc)

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")

        # Plot metrics
        metrics.plot_metrics(save_dir)

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # ckpt_path = save_dir / f"{CONFIG['model_save_prefix']}_trial_{trial+1}_epoch{epoch+1}.pt"
            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            torch.save({
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'metrics': {
                    'train_losses': metrics.train_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_losses': metrics.val_losses,
                    'val_accuracies': metrics.val_accuracies
                }
            }, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")

    print("Training complete!")

In [ ]:
# Loading a saved model
ckpt_path = experiment_config.checkpoint_path
# ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
checkpoint = torch.load(ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

# if 'mode' not in checkpoint['config']:
    # checkpoint['config']['mode'] = "LM"  # Default to LM if not specified

if checkpoint['config'].model_config.mode == 'LM':
    model = TinyLlamaTransformer(
        checkpoint['config'].model_config
    ).to(experiment_config.device)

elif checkpoint['config'].model_config.mode == 'RLM':
        model = TinyLlamaRawTransformer(
            embed_dim=CONFIG['embed_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window']
        ).to(CONFIG['device'])

elif checkpoint['config'].model_config.mode == 'AE':
    model = TransformerAutoencoder(
            vocab_size=checkpoint['config']['vocab_size'],
            embed_dim=checkpoint['config']['embed_dim'],
            latent_dim=checkpoint['config']['latent_dim'],
            n_layers=checkpoint['config']['n_layers'],
            n_heads=checkpoint['config']['n_heads'],
            ffn_dim=checkpoint['config']['ffn_dim'],
            context_window=checkpoint['config']['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=checkpoint['config']['n_latents']
        ).to(CONFIG['device'])
elif checkpoint['config'].model_config.mode == 'KAE':
    model = TransformerKoopmanAutoencoder(
            vocab_size=CONFIG['vocab_size'],
            embed_dim=CONFIG['embed_dim'],
            latent_dim=CONFIG['latent_dim'],
            n_layers=CONFIG['n_layers'],
            n_heads=CONFIG['n_heads'],
            ffn_dim=CONFIG['ffn_dim'],
            context_window=CONFIG['context_window'],
            cls_id=cls_id,
            sos_id=sos_id,
            n_latents=CONFIG['n_latents'],
            n_diagonals=CONFIG['n_diagonals']
        ).to(CONFIG['device'])


model_state = checkpoint['model_state_dict']
model.load_state_dict(model_state)

In [ ]:
prompt = ['E1', "M1", "N1"]

max_new_tokens = 80
temperature = 1
top_k = 10

generated, probs = generate_from_model(
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    prompt=prompt,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

print(generated)

In [ ]:

fig = plot_generation_probabilities(
    probabilities=probs,
    sequence=generated[len(prompt):],
    token2id=NT_token2id,
    id2token=NT_id2token,
    tokens_to_highlight=None,
    figsize=(12, 8)
)

In [ ]:
# generate trajectories for DVAE training

context_window = experiment_config.model_config.context_window
n_train = 200
n_train_windows = 1
max_steps = n_train_windows * (context_window + 1)
n_val = 50
train_dataset, train_dataset_NT, train_seen = generate_tiny_memory_dataset(n_samples=n_train, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
val_dataset, val_dataset_NT, val_seen = generate_tiny_memory_dataset(n_samples=n_val, seen=train_seen, P_transitions=P_transitions, NT_vocab=NT_vocab,NT_token2id=NT_token2id, NT_to_T=NT_to_T, num_steps=max_steps, memory_limit=memory_limit)
# train_blocks = prepare_blocks(train_dataset_NT, NT_token2id, context_window)
# val_blocks = prepare_blocks(val_dataset_NT, NT_token2id, context_window)

max_new_tokens = 50
temperature = 1
top_k = 10
trajectories_per_initial_seq = 5

# returns a np.array of encoded trajectories, shape [B, T]
train_trajectories = create_stacked_trajectories_array(
    initial_seqs=train_dataset_NT,
    context_window=context_window,
    trajectories_per_initial_seq=trajectories_per_initial_seq,
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

val_trajectories = create_stacked_trajectories_array(
    initial_seqs=val_dataset_NT,
    context_window=context_window,
    trajectories_per_initial_seq=trajectories_per_initial_seq,
    model=model,
    token2id=NT_token2id,
    id2token=NT_id2token,
    max_new_tokens=max_new_tokens,
    temperature=temperature,
    top_k=top_k,
    device=experiment_config.device
)

In [ ]:
# print(train_trajectories[0])
# print(train_trajectories[1])
# print([NT_id2token[id] for id in train_trajectories[0]])
# print([NT_id2token[id] for id in train_trajectories[1]])

In [ ]:
# Save trajectories
np.save('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_1000_tokenized_trajectories_flat.npy', train_trajectories)
np.save('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_250_tokenized_trajectories_flat.npy', val_trajectories)

In [ ]:
# Load trajectories
train_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_train_context_window_32_num_steps_50_1000_tokenized_trajectories_flat.npy')
val_trajectories = np.load('/home/galk/LanguageDynamics/data/tiny_memory_val_context_window_32_num_steps_50_250_tokenized_trajectories_flat.npy')

In [ ]:
# DVAE config
model_config = TinyDVAEConfig(
    n_layers=4,
    n_heads=4,
    embed_dim=32,
    ffn_dim=256,
    context_window=32,
    vocab=NT_vocab,
    dropout_self_attention=0.05,
    dropout_embed=0.03,
    dropout_residual=0.03,
    dropout_latent=0.00,
    latent_dim=5,
    decoder_ffn_dim=64,
    dropout_decoder=0.03,
    transition_ffn_dim=64,
    dropout_transition=0.03,
    pooling='last',
    decoder_ln=False,
    transition_ln=False
)

training_config = TrainingDVAEConfig(
    lr=1e-3,
    batch_size=64,
    grad_clipping=True,
    reconstruction_coef=1.0,
    warmup_steps=1000,
    minimal_beta=0.0,
    maximal_beta=1.0,
    n_encoder_layers=4,
    teacher_forcing=True,
    use_scheduler=True,
    scheduler_T_max=500,
    scheduler_eta_min=1e-5
)

experiment_config = ExperimentConfig(
    epochs=500,
    checkpoint_path=None,
    # checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt",
    save_every=10,
    device_index=0,
    model_config=model_config,
    training_config=training_config,
    LM_checkpoint_path="/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_LM_dyck2_layers_4_embed_32_ffn_dim_256_context_window_32_date_110925_1816_trial_1/ckpt_epoch2.pt"
)

pad_id = 100

# TODO: move the model save prefix to the ExperimentConfig initialization
current_date_ddmmyy = datetime.now().strftime('%d%m%y_%H%M')
if model_config.mode == 'AE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'KAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_n_latents_{model_config.n_latents}_n_diagonals_{model_config.n_diagonals}_maxdepth_{model_config.max_depth}_maxlen_{model_config.max_length}_date_{current_date_ddmmyy}"
if model_config.mode == 'DVAE':
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_latent_{model_config.latent_dim}_date_{current_date_ddmmyy}"
elif model_config.mode == 'LM' or model_config.mode == 'RLM':
    # TODO: add memory generation parameters to name
    experiment_config.model_save_prefix = f"tiny_{model_config.mode}_tinymemory_layers_{model_config.n_layers}_embed_{model_config.embed_dim}_ffn_dim_{model_config.ffn_dim}_context_window_{model_config.context_window}_date_{current_date_ddmmyy}"

print(f"Using device: {experiment_config.device}")

In [ ]:
train_trajectory_dataset = TrajectoryDataset(train_trajectories)
val_trajectory_dataset = TrajectoryDataset(val_trajectories)

In [ ]:
def collate_fn_np(batch):
    # 'batch' is a list of np arrays, of shape [T] of trajectory IDs
    trajs = np.array(batch) # [B, T]
    trajs = torch.tensor(trajs, dtype=torch.long)  # shape [B, T]
    return trajs

train_loader = DataLoader(
    train_trajectory_dataset, 
    batch_size=experiment_config.training_config.batch_size, 
    shuffle=True, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    # num_workers=16,
    # prefetch_factor=2,
    # persistent_workers=True,
    # pin_memory=True
    )

val_loader = DataLoader(
    val_trajectory_dataset, 
    # batch_size=experiment_config.training_config.batch_size, 
    batch_size=experiment_config.training_config.batch_size,
    shuffle=False, 
    collate_fn=collate_fn_np, 
    drop_last=True,
    # num_workers=16,
    # prefetch_factor=2,
    # persistent_workers=True,
    # pin_memory=True
    )


In [ ]:
### Dynamical VAE Training
import matplotlib.pyplot as plt
from datetime import datetime
import time
from pathlib import Path
from torch.distributions.kl import kl_divergence
from torch.distributions.multivariate_normal import MultivariateNormal

def calculate_accuracy(logits, targets, pad_id):
    # Flatten predictions and targets
    pred = logits.argmax(dim=-1).view(-1)
    true = targets.view(-1)
    # Create mask to ignore padding tokens
    mask = (true != pad_id)
    # Calculate accuracy only on non-pad tokens
    correct = (pred[mask] == true[mask]).float().sum()
    total = mask.sum()
    return (correct / total).item() if total > 0 else 0

class MetricsTracker:
    def __init__(self):
        # Training metrics
        self.train_total_losses = []
        self.train_reconstruction_losses = []
        self.train_kl_losses = []
        self.train_accuracies = []
        
        # Validation metrics
        self.val_reconstruction_losses = []
        self.val_kl_losses = []
        self.val_accuracies = []
        
        self.epochs = []
        self.learning_rates = []

    def load_from_checkpoint(self, metrics_dict):
        """Load metrics from a checkpoint dictionary"""
        self.train_total_losses = metrics_dict.get('train_total_losses', [])
        self.train_reconstruction_losses = metrics_dict.get('train_reconstruction_losses', [])
        self.train_kl_losses = metrics_dict.get('train_kl_losses', [])
        self.train_accuracies = metrics_dict.get('train_accuracies', [])
        
        self.val_reconstruction_losses = metrics_dict.get('val_reconstruction_losses', [])
        self.val_kl_losses = metrics_dict.get('val_kl_losses', [])
        self.val_accuracies = metrics_dict.get('val_accuracies', [])
        self.learning_rates = metrics_dict.get('learning_rates', [])
        
        self.epochs = list(range(1, len(self.train_total_losses) + 1))

    def plot_metrics(self, save_dir=None):
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Color scheme
        train_color = "#287BBB"  # Blue
        val_color = "#ED9E00"    # Orange
        
        # Plot 1: Reconstruction losses
        axes[0,0].plot(self.epochs, self.train_reconstruction_losses, 
                    color=train_color, linestyle='-', label='Train Reconstruction')
        axes[0,0].plot(self.epochs, self.val_reconstruction_losses, 
                    color=val_color, linestyle='-', label='Val Reconstruction')
        axes[0,0].set_xlabel('Epoch')
        axes[0,0].set_ylabel('Loss')
        axes[0,0].set_title('Reconstruction Loss')
        axes[0,0].legend()
        axes[0,0].grid(True)

        # Plot 2: KL losses
        axes[0,1].plot(self.epochs, self.train_kl_losses, 
                    color=train_color, linestyle='-', label='Train KL')
        axes[0,1].plot(self.epochs, self.val_kl_losses, 
                    color=val_color, linestyle='-', label='Val KL')
        axes[0,1].set_xlabel('Epoch')
        axes[0,1].set_ylabel('Loss')
        axes[0,1].set_title('KL Divergence Loss')
        axes[0,1].legend()
        axes[0,1].grid(True)

        # Plot 3: Total training loss and LR
        ax3_twin = axes[1,0].twinx()
        axes[1,0].plot(self.epochs, self.train_total_losses, color=train_color, label='Total Train Loss')
        ax3_twin.plot(self.epochs, self.learning_rates, color='green', linestyle='--', label='Learning Rate')
        axes[1,0].set_xlabel('Epoch')
        axes[1,0].set_ylabel('Loss')
        ax3_twin.set_ylabel('Learning Rate')
        axes[1,0].set_title('Total Training Loss and Learning Rate')
        axes[1,0].legend(loc='upper left')
        ax3_twin.legend(loc='upper right')
        axes[1,0].grid(True)


        # Plot 4: Accuracies
        axes[1,1].plot(self.epochs, self.train_accuracies, 
                    color=train_color, linestyle='-', label='Train Accuracy')
        axes[1,1].plot(self.epochs, self.val_accuracies, 
                    color=val_color, linestyle='-', label='Val Accuracy')
        axes[1,1].set_xlabel('Epoch')
        axes[1,1].set_ylabel('Accuracy')
        axes[1,1].set_title('Model Accuracy')
        axes[1,1].legend()
        axes[1,1].grid(True)

        plt.tight_layout()

        if save_dir:
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            plt.savefig(f"{save_dir}/metrics_{timestamp}.png", dpi=300, bbox_inches='tight')

        plt.show()

def load_checkpoint(checkpoint_path, model, optimizer, scheduler, experiment_config, device):
    """
    Load model and optimizer state from a checkpoint
    Returns the starting epoch, loaded metrics, and global step
    """
    print(f"Loading checkpoint from {checkpoint_path}")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Load model state
    model.load_state_dict(checkpoint['model_state_dict'])

    # Load optimizer state if provided
    if optimizer is not None:
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    
    # Load scheduler state if provided
    if scheduler is not None and 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

    # Verify config matches
    saved_config = checkpoint['config']
    # Compare relevant parts of the model config
    current_model_config = experiment_config.model_config
    saved_model_config = saved_config.model_config
    
    mismatched_keys = []
    for key in ['n_layers', 'n_heads', 'embed_dim', 'ffn_dim', 'context_window', 'latent_dim']:
        if getattr(current_model_config, key) != getattr(saved_model_config, key):
            mismatched_keys.append(key)
    
    if mismatched_keys:
        raise ValueError(f"Checkpoint config mismatch for keys: {mismatched_keys}")

    starting_epoch = checkpoint['epoch']
    global_step = checkpoint.get('global_step', 0)
    return starting_epoch, checkpoint['metrics'], global_step

n_trials = 1
for trial in range(n_trials):
    # Initialize model
    if experiment_config.model_config.mode == 'DVAE':
        model = TransformerDVAE(
            experiment_config.model_config
        ).to(experiment_config.device)

    else:
        raise ValueError(f"Unsupported mode: {experiment_config.model_config.mode}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=experiment_config.training_config.lr)
    scheduler = None
    if experiment_config.training_config.use_scheduler:
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=experiment_config.training_config.scheduler_T_max,
            eta_min=experiment_config.training_config.scheduler_eta_min
        )
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    metrics = MetricsTracker()

    # Create save directory
    save_dir = Path(os.path.join(models_path, experiment_config.model_save_prefix + f"_trial_{trial+1}"))
    save_dir.mkdir(exist_ok=True, parents=True)

    # Load checkpoint if specified
    starting_epoch = 0
    global_step = 0
    if experiment_config.checkpoint_path:
        try:
            starting_epoch, saved_metrics, global_step = load_checkpoint(
                experiment_config.checkpoint_path,
                model,
                optimizer,
                scheduler,
                experiment_config,
                experiment_config.device
            )
            for param_group in optimizer.param_groups:
                param_group['lr'] = experiment_config.training_config.lr
            metrics.load_from_checkpoint(saved_metrics)
            print(f"Resuming training from epoch {starting_epoch}, global_step {global_step}")
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting training from scratch")
            starting_epoch = 0
            global_step = 0

    print("Starting training loop...")
    global_start = time.time()

    # Load a pretrained LM model to use its encoder
    # Loading a saved model
    lm_ckpt_path = experiment_config.LM_checkpoint_path
    # ckpt_path = "/content/tiny_transformer_dyck2_layers_2_embed_32_context_window_64_weight_tied_maxdepth_20_max_len_50_date_29_07_25_trial_1/model_epoch15.pt"
    if lm_ckpt_path:
        LM_checkpoint = torch.load(lm_ckpt_path, map_location=torch.device(experiment_config.device), weights_only=False)

        if LM_checkpoint['config'].model_config.mode == 'LM':
            lm_model = TinyLlamaTransformer(
                LM_checkpoint['config'].model_config
            ).to(experiment_config.device)
        model.encoder = lm_model
        for param in model.encoder.parameters():
            param.requires_grad = False

    def validate(model: TransformerDVAE, val_loader, criterion, device):
        model.eval()
        total_reconstruction_loss = 0
        total_kl_loss = 0
        total_acc = 0
        with torch.no_grad():
            for x in val_loader:
                x = x.to(device)
                
                # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
                B, T_data = x.shape
                C = model.context_window
                x_unfolded = x.unfold(dimension=-1, size=C, step=1)
                B, T, C_out = x_unfolded.shape
                
                x_reshaped = x_unfolded.reshape(B * T, C_out)
                
                # inference: encode observations into latents
                z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

                # decode the latents to reconstruct the observations
                logits = model.decode(z) # logits: [B*T, vocab_size]

                loss_reconstruction = criterion(logits, x_reshaped[:, -1])
                acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)

                # reshape z to [B, T, latent_dim] for KL calculation
                z = z.reshape(B, T, model.latent_dim)
                
                # initial_latent = torch.zeros((B, model.latent_dim), device=device)
                # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
                # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
                z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
                mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

                # # concatenate t=0 prior
                # mu_p = mu_p.reshape(B, T-1, -1)
                # logvar_p = logvar_p.reshape(B, T-1, -1)
                # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
                # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

                # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
                mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
                logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

                # Compute KL divergence loss
                Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
                loss_kl = Dkl.sum() / (B * (T-1) * model.latent_dim)

                total_reconstruction_loss += loss_reconstruction.item()
                total_kl_loss += loss_kl.item()
                total_acc += acc

                # predictions distributional statistics
                mean_mu_q = mu_q.mean(dim=0).detach().cpu()
                std_mu_q = mu_q.std(dim=0).detach().cpu()
                mean_logvar_q = logvar_q.mean(dim=0).detach().cpu()
                std_logvar_q = logvar_q.std(dim=0).detach().cpu()
                mean_mu_p = mu_p.mean(dim=0).detach().cpu()
                std_mu_p = mu_p.std(dim=0).detach().cpu()
                mean_logvar_p = logvar_p.mean(dim=0).detach().cpu()
                std_logvar_p = logvar_p.std(dim=0).detach().cpu()

                correlation_matrix = torch.corrcoef(torch.stack((mean_mu_q, mean_mu_p)))

                # The Pearson correlation coefficient between x and y is at index [0, 1] or [1, 0]
                pearson_r = correlation_matrix[0, 1]

                # print(f"val batch:")
                # print(f"mean_mu_q={mean_mu_q}, \nstd_mu_q={std_mu_q}, \nmean_logvar_q={mean_logvar_q}, \nstd_logvar_q={std_logvar_q}")
                # print(f"mean_mu_p={mean_mu_p}, \nstd_mu_p={std_mu_p}, \nmean_logvar_p={mean_logvar_p}, \nstd_logvar_p={std_logvar_p}")
                # print(f"mean_mu correlation: {pearson_r}")

        return total_reconstruction_loss / len(val_loader), total_kl_loss / len(val_loader), total_acc / len(val_loader)

    # Training loop
    device = experiment_config.device
    for epoch in range(starting_epoch, experiment_config.epochs):
        model.train()
        if lm_ckpt_path:
            model.encoder.eval()  # keep LM encoder in eval mode
        epoch_loss = 0.0
        epoch_loss_reconstruction = 0.0
        epoch_loss_kl = 0.0
        epoch_acc = 0.0
        start = time.time()

        # Training phase
        for batch, x in enumerate(tqdm(train_loader, desc=f"Epoch {epoch+1} Training")):
            x = x.to(device)
            
            # x is [B, T] tensor of token trajectories. Virtually stack it into [B, T', C] windows
            B, T_data = x.shape
            C = model.context_window
            x_unfolded = x.unfold(dimension=-1, size=C, step=1)
            B, T, C_out = x_unfolded.shape
            
            x_reshaped = x_unfolded.reshape(B * T, C_out)
            
            # inference: encode observations into latents
            z, mu_q, logvar_q = model.inference(x_reshaped, n_layers=experiment_config.training_config.n_encoder_layers) # z, mu, logvar: [B*T, latent_dim]

            # decode the latents to reconstruct the observations
            logits = model.decode(z) # logits: [B*T, vocab_size]

            loss_reconstruction = criterion(logits, x_reshaped[:, -1]) # reconstruction loss only for the last token in each window
            acc = calculate_accuracy(logits, x_reshaped[:, -1], pad_id)  # autoencoding reconstruction accuracy

            # reshape z to [B, T, latent_dim] for KL calculation
            z = z.reshape(B, T, model.latent_dim)

            # # TODO: add option to sample from initial prior instead of zero
            # initial_latent = torch.zeros((B, model.latent_dim), device=device)
            # mu_p_t0, logvar_p_t0 = model.transition_model(initial_latent, is_t0=True)  # [B, latent_dim]
            # z_prev = torch.cat([initial_latent.unsqueeze(1), z[:, :-1, :]], dim=1)  # [B, T, latent_dim]
            z_prev = z[:, :-1, :]  # [B, T-1, latent_dim]
            mu_p, logvar_p = model.transition_model(z_prev.reshape(B*(T-1), -1))  # [B*(T-1), latent_dim]

            # # concatenate t=0 prior
            # mu_p = mu_p.reshape(B, T-1, -1)
            # logvar_p = logvar_p.reshape(B, T-1, -1)
            # mu_p = torch.cat([mu_p_t0.unsqueeze(1), mu_p], dim=1).reshape(B*T, -1)
            # logvar_p = torch.cat([logvar_p_t0.unsqueeze(1), logvar_p], dim=1).reshape(B*T, -1)

            # remove first posterior from each trajectory (it cannot be predicted) to match prior shape
            mu_q = mu_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)
            logvar_q = logvar_q.reshape(B, T, -1)[:, 1:, :].reshape(B*(T-1), -1)

            dist_q = MultivariateNormal(loc=mu_q, covariance_matrix=torch.diag_embed(torch.exp(logvar_q)))
            dist_p = MultivariateNormal(loc=mu_p, covariance_matrix=torch.diag_embed(torch.exp(logvar_p)))

            KL_dist = kl_divergence(dist_q, dist_p)

            # Compute KL divergence loss
            # Dkl = kl_divergence_gaussians(mu_q, logvar_q, mu_p, logvar_p)  # [B*(T-1)]
            loss_kl = KL_dist.sum() / (B * (T-1) * model.latent_dim)

            # print(f"torch.kl: {KL_dist}, my KL: {Dkl}, KL loss: {loss_kl}")

            # compute KL-annealing beta
            beta = (experiment_config.training_config.maximal_beta - experiment_config.training_config.minimal_beta) * min(1.0, global_step / experiment_config.training_config.warmup_steps) + experiment_config.training_config.minimal_beta
            loss = experiment_config.training_config.reconstruction_coef * loss_reconstruction + beta * loss_kl

            optimizer.zero_grad()
            loss.backward()

            # Optional: Gradient clipping
            if experiment_config.training_config.grad_clipping:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            global_step += 1

            epoch_loss += loss.item()
            epoch_loss_reconstruction += loss_reconstruction.item()
            epoch_loss_kl += loss_kl.item()
            epoch_acc += acc

        # Validation phase
        val_reconstruction_loss, val_kl_loss, val_acc = validate(model, val_loader, criterion, device)

        # Calculate average losses and accuracies
        train_loss = epoch_loss / len(train_loader)
        train_acc = epoch_acc / len(train_loader)
        train_loss_reconstruction = epoch_loss_reconstruction / len(train_loader)
        train_loss_kl = epoch_loss_kl / len(train_loader)

        # Update metrics
        metrics.epochs.append(epoch + 1)
        metrics.train_total_losses.append(train_loss)
        metrics.train_reconstruction_losses.append(train_loss_reconstruction)
        metrics.train_kl_losses.append(train_loss_kl)
        metrics.train_accuracies.append(train_acc)
        metrics.val_reconstruction_losses.append(val_reconstruction_loss)
        metrics.val_kl_losses.append(val_kl_loss)
        metrics.val_accuracies.append(val_acc)
        metrics.learning_rates.append(optimizer.param_groups[0]['lr'])

        end = time.time()

        # Print metrics
        print(f"\n[Epoch {epoch+1}]")
        print(f"Train Total Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Train Reconstruction Loss: {train_loss_reconstruction:.4f} | Train KL Loss: {train_loss_kl:.4f}")
        print(f"Val Reconstruction Loss: {val_reconstruction_loss:.4f} | Val KL Loss: {val_kl_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Epoch time: {end-start:.1f}s | Total time: {end-global_start:.1f}s")
        
        # Step the scheduler
        if scheduler:
            scheduler.step()
            print(f"LR: {scheduler.get_last_lr()[0]:.6f}")

        # Save checkpoint
        if (epoch+1) % experiment_config.save_every == 0 or (epoch+1) == experiment_config.epochs:
            # Plot metrics
            metrics.plot_metrics(save_dir)

            ckpt_path = save_dir / f"ckpt_epoch{epoch+1}.pt"
            checkpoint_data = {
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'config': experiment_config,
                'epoch': epoch+1,
                'global_step': global_step,
                'metrics': {
                    'train_total_losses': metrics.train_total_losses,
                    'train_reconstruction_losses': metrics.train_reconstruction_losses,
                    'train_kl_losses': metrics.train_kl_losses,
                    'train_accuracies': metrics.train_accuracies,
                    'val_reconstruction_losses': metrics.val_reconstruction_losses,
                    'val_kl_losses': metrics.val_kl_losses,
                    'val_accuracies': metrics.val_accuracies,
                    'learning_rates': metrics.learning_rates
                }
            }
            if scheduler:
                checkpoint_data['scheduler_state_dict'] = scheduler.state_dict()
            
            torch.save(checkpoint_data, ckpt_path)
            print(f"Saved checkpoint: {ckpt_path}")


    print("Training complete!")

In [ ]:
logvar_q.shape

In [ ]:
checkpoint_path = "/home/galk/LanguageDynamics/models/linguistic_flip_flop/tiny_DVAE_tinymemory_layers_4_embed_32_ffn_dim_256_context_window_32_latent_5_date_240925_1307_trial_1/ckpt_epoch360.pt"
checkpoint = torch.load(checkpoint_path, map_location=torch.device(experiment_config.device), weights_only=False)
model = TransformerDVAE(
    experiment_config.model_config
).to(experiment_config.device)

model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
# test model reconstruction
x = val_trajectories[0]  # [T] numpy array of token
x = torch.tensor(x, dtype=torch.long).unsqueeze(0).to(experiment_config.device)  # [1, T]
C = model.context_window
x_unfolded = x.unfold(dimension=-1, size=C, step=1)
B, T, C_out = x_unfolded.shape
x_reshaped = x_unfolded.reshape(B * T, C_out)
model.eval()
with torch.no_grad():
    z, mu_q, logvar_q = model.inference(x_reshaped) # z, mu, logvar: [1*T, latent_dim]
    logits = model.decode(z) # logits: [1*T, vocab_size]
    reconstructed_ids = logits.argmax(dim=-1).cpu().numpy()  # [1*T]
    reconstructed_tokens = [NT_id2token[id] for id in reconstructed_ids]
    print("Original:     ", [NT_id2token[id] for id in x[0, C-1:].cpu().numpy()])
    print("Reconstructed:", reconstructed_tokens)

In [ ]:
z0 = z[:1]

In [ ]:
latent_trajectory, decoded_logits = model.generate_latent_trajectory(seq_len=100, z0=z0, do_reparameterization=True, device=experiment_config.device)

In [ ]:
# decode logits to tokens
decoded_ids = decoded_logits.argmax(dim=-1).cpu().numpy()  # [B, seq_len]

decoded_tokens = [NT_id2token[id] for id in decoded_ids[0]]

In [ ]:
print(decoded_tokens)